# 09 — Explainability

## Objective

This notebook explains the exact fitted standard-Python model saved by Notebook 08. It no longer substitutes a Logistic Regression surrogate when Random Forest or XGBoost is selected.


In [ ]:
from __future__ import annotations

import importlib.util
import json
import os
from pathlib import Path

bootstrap = Path.cwd() / "notebooks" / "import_path.py"
if not bootstrap.is_file():
    bootstrap = Path.cwd() / "import_path.py"
spec = importlib.util.spec_from_file_location("import_path", bootstrap)
ip = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ip)

import joblib
import numpy as np
import pandas as pd
import shap

from pyspark.sql import functions as F
from config import project_config as cfg

RANDOM_SEED = cfg.RANDOM_SEED

def filesystem_path(path_value):
    path_value = str(path_value)
    return "/dbfs/" + path_value[6:] if path_value.startswith("dbfs:/") else path_value

model_bundle_path = os.path.join(filesystem_path(cfg.SELECTED_MODEL_PATH), "model_bundle.joblib")
bundle = joblib.load(model_bundle_path)
pipeline = bundle["pipeline"]
model_name = bundle["model_name"]
manifest = bundle["feature_manifest"]
input_columns = list(manifest["model_input_columns"])

print(f"Explaining fitted model: {model_name}")


## Load a Reproducible Explanation Sample

The validation data are used for model interpretation because they follow the training period and retain their natural class distribution. The sample is bounded to keep SHAP computation practical.


In [ ]:
sample_size = int(getattr(cfg, "SHAP_SAMPLE_SIZE", 1_000))
explanation_spark_df = (
    spark.table(cfg.MODELING_VALIDATION_HIST_TABLE)
    .select(*input_columns, cfg.TARGET_COLUMN)
    .orderBy(F.rand(RANDOM_SEED))
    .limit(sample_size)
)
explanation_pdf = explanation_spark_df.toPandas()
X_explain = explanation_pdf[input_columns]

preprocessor = pipeline.named_steps["preprocessor"]
classifier = pipeline.named_steps["classifier"]
X_transformed = preprocessor.transform(X_explain)
feature_names = np.asarray(preprocessor.get_feature_names_out(), dtype=str)

print(f"Explanation rows: {len(explanation_pdf):,}")
print(f"Transformed features: {len(feature_names):,}")


## Compute Model-Specific SHAP Values

Linear SHAP is used for Logistic Regression, while Tree SHAP is used for Random Forest and XGBoost. The resulting importance values therefore describe the selected model itself.


In [ ]:
def dense_if_needed(matrix):
    return matrix.toarray() if hasattr(matrix, "toarray") else np.asarray(matrix)


if model_name == "Logistic Regression":
    transformed_dense = dense_if_needed(X_transformed)
    background = shap.sample(transformed_dense, min(200, len(transformed_dense)), random_state=RANDOM_SEED)
    explainer = shap.LinearExplainer(classifier, background)
    shap_values = explainer(transformed_dense)
    shap_array = np.asarray(shap_values.values)
elif model_name in {"Random Forest", "XGBoost"}:
    explainer = shap.TreeExplainer(classifier)
    raw_values = explainer.shap_values(X_transformed)
    if isinstance(raw_values, list):
        shap_array = np.asarray(raw_values[-1])
    else:
        shap_array = np.asarray(raw_values)
        if shap_array.ndim == 3:
            shap_array = shap_array[:, :, -1]
else:
    raise ValueError(f"Unsupported model for SHAP: {model_name}")

if shap_array.shape[1] != len(feature_names):
    raise ValueError("SHAP feature dimension does not match the fitted preprocessor.")

mean_absolute = np.mean(np.abs(shap_array), axis=0)
mean_signed = np.mean(shap_array, axis=0)
importance_pdf = pd.DataFrame({
    "Feature": feature_names,
    "MeanAbsSHAP": mean_absolute,
}).sort_values("MeanAbsSHAP", ascending=False)

direction_pdf = pd.DataFrame({
    "Feature": feature_names,
    "Mean_SHAP": mean_signed,
    "Mean_Absolute_SHAP": mean_absolute,
    "Effect_Direction": np.where(mean_signed >= 0, "Increases delay risk", "Decreases delay risk"),
}).sort_values("Mean_Absolute_SHAP", ascending=False)

display(spark.createDataFrame(importance_pdf.head(30)))


In [ ]:
shap.summary_plot(
    shap_array,
    dense_if_needed(X_transformed),
    feature_names=feature_names,
    max_display=20,
    show=True,
)


## Persist Global, Directional, and Local Explanations

The table names and schemas are preserved for Notebooks 10 and 11. Local explanations use row indices from the reproducible explanation sample.


In [ ]:
(
    spark.createDataFrame(importance_pdf)
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(cfg.SHAP_GLOBAL_IMPORTANCE_TABLE)
)
(
    spark.createDataFrame(direction_pdf)
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(cfg.SHAP_DIRECTION_EFFECTS_TABLE)
)

top_feature_count = int(getattr(cfg, "SHAP_VALUES_TOP_FEATURES", 15))
sample_row_count = min(int(getattr(cfg, "SHAP_VALUES_SAMPLE_ROWS", 100)), len(shap_array))
top_indices = np.argsort(-mean_absolute)[:top_feature_count]
local_rows = []
for row_index in range(sample_row_count):
    for feature_index in top_indices:
        local_rows.append({
            "row_index": int(row_index),
            "feature": str(feature_names[feature_index]),
            "shap_value": float(shap_array[row_index, feature_index]),
        })

local_pdf = pd.DataFrame(local_rows)
(
    spark.createDataFrame(local_pdf)
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(cfg.SHAP_VALUES_SAMPLE_TABLE)
)

local_path = filesystem_path(cfg.SHAP_LOCAL_EXPLANATION_PATH)
os.makedirs(os.path.dirname(local_path), exist_ok=True)
with open(local_path, "w", encoding="utf-8") as handle:
    json.dump({
        "model_name": model_name,
        "sample_rows": sample_row_count,
        "top_features": importance_pdf.head(top_feature_count).to_dict("records"),
    }, handle, indent=4)

print("Actual selected-model SHAP artifacts saved successfully.")


## Explainability Conclusion

These explanations are aligned with the actual final candidate and its fitted preprocessing pipeline. They should be interpreted alongside holdout performance and calibration from Notebook 08, rather than as standalone evidence of model quality.
